# 3 · Wer wird still? Ein Churn-Modell
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Bei Päckli & Co. kündigt niemand: Kunden hören einfach auf zu kaufen. Wir legen fest:
**abgewandert = keine Bestellung in den nächsten 90 Tagen.** Das Modell lernt aus der Vergangenheit
(Stichtag 31. Mai 2025) und bewertet dann alle Kunden zum 31. August 2025.

**Minimum:** ein Risikowert pro Kunde (`churn_scores`), Lift der oberen 20 % und der Vergleich mit der Handregel.
**Erweiterung (optional):** Entscheidungsbaum, Lift-Kurve, weitere Merkmale.

**Leakage:** Merkmale dürfen nur Daten **vor** dem Stichtag benutzen. Sonst sieht das Modell im Test brillant aus und versagt im Einsatz.

**So arbeitet ihr in jedem Schritt**
1. Lest das **Ziel**.
2. Kopiert den **Prompt** ins Gemini-Fenster (Gemini-Symbol oben rechts) und lasst euch den Code erklären.
3. Fügt den Code in die leere Zelle ein und führt sie aus (▶ links oder `Shift + Enter`).
4. **Plausibel?** Prüft das Ergebnis mit gesundem Menschenverstand.
5. Steckt ihr fest, führt die **💡 Lösung** aus. Das ist keine Schande: Die Lösung liefert dieselben Tabellen wie euer Code.

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, urllib.request
if not os.path.exists("kurs.py"):
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

In [ ]:
#@title ▶ Aufholen: Ergebnisse aus Block 1 laden (eure eigenen, sonst das Musterergebnis)
orders_clean = aufholen("orders_clean")
rfm = aufholen("rfm")

### 0 Aufwärmen: Segmente ohne Label (Live-Demo)

**Ziel:** Unüberwachtes Lernen auf euren eigenen Daten: die RFM-Tabelle in vier Gruppen teilen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Teile die Kunden in rfm mit k-Means in 4 Segmente. Nimm den Logarithmus (np.log1p) von tage_seit_kauf, frequenz und umsatz und standardisiere sie. Zeige pro Segment die Anzahl Kunden und die Mittelwerte.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(np.log1p(rfm[["tage_seit_kauf", "frequenz", "umsatz"]]))
rfm_seg = rfm.assign(segment=KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X))
rfm_seg.groupby("segment").agg(kunden=("customer_id", "size"), tage_seit_kauf=("tage_seit_kauf", "mean"),
                               frequenz=("frequenz", "mean"), umsatz=("umsatz", "mean")).round(0)

**Plausibel?** Ergeben die Segmente Sinn? Hier gibt es keine richtige Antwort, die man prüfen könnte: Das ist der Unterschied zum Churn-Modell, das ihr gleich baut.

### 1 Das Label: Wer ist abgewandert?

**Ziel:** Für jeden Kunden, der vor dem Stichtag gekauft hat, festhalten: Hat er in den 90 Tagen danach bestellt?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Erstelle aus orders_clean einen DataFrame label mit allen Kunden, die vor dem 1. Juni 2025 mindestens einmal bestellt haben, und einer Spalte churn: 1, wenn der Kunde vom 1. Juni 2025 bis und mit 29. August 2025 keine Bestellung hat, sonst 0. Wie hoch ist der Anteil churn = 1?
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
label = duckdb.sql("""
SELECT customer_id,
       CASE WHEN customer_id IN (SELECT customer_id FROM orders_clean
                                 WHERE invoice_date >= DATE '2025-06-01' AND invoice_date < DATE '2025-08-30')
            THEN 0 ELSE 1 END AS churn
FROM orders_clean
WHERE invoice_date < DATE '2025-06-01'
GROUP BY customer_id
""").df()
print(f"{len(label)} Kunden, davon abgewandert: {label['churn'].mean():.0%}")

**Plausibel?** Ein hoher Anteil? Viele «Abgewanderte» haben nur einmal gekauft. Waren das je richtige Kunden? Und: Wäre die Zahl mit 30 oder 180 Tagen anders? Die Definition ist eine Business-Entscheidung.

### 2 Merkmale vor dem Stichtag

**Ziel:** RFM-Merkmale berechnen, aber nur aus Bestellungen vor dem Stichtag. Als Funktion, damit ihr sie später für den 31. August wiederverwenden könnt.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Schreibe eine Funktion merkmale(stichtag), die aus orders_clean nur Bestellungen bis und mit stichtag nimmt und pro Kunde tage_seit_kauf (bis zum stichtag), frequenz und umsatz berechnet. Berechne train_df = merkmale('2025-05-31') und hänge die Spalte churn aus label an.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
def merkmale(stichtag):
    return duckdb.sql(f"""
    SELECT customer_id,
           date_diff('day', MAX(invoice_date)::DATE, DATE '{stichtag}') AS tage_seit_kauf,
           COUNT(DISTINCT invoice)                                      AS frequenz,
           SUM(quantity * price)                                        AS umsatz
    FROM orders_clean
    WHERE invoice_date < DATE '{stichtag}' + INTERVAL 1 DAY
    GROUP BY customer_id
    """).df()

train_df = merkmale("2025-05-31").merge(label, on="customer_id")
train_df.describe().round(1)

**Plausibel?** Leakage-Check: Ist `tage_seit_kauf` nie negativ? Hat `train_df` gleich viele Zeilen wie `label`? Ein Merkmal wie «Bestellungen im Juni» wäre verboten: Das ist die Zukunft.

In [ ]:
print("Zeilen:", len(train_df), "| kleinste tage_seit_kauf:", train_df["tage_seit_kauf"].min())

### 3 Das Modell

**Ziel:** Eine logistische Regression lernt aus den Beispielen, wie Merkmale und Abwanderung zusammenhängen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Trainiere mit scikit-learn eine logistische Regression, die churn aus tage_seit_kauf, frequenz und umsatz in train_df vorhersagt. Nimm den Logarithmus (np.log1p) der Merkmale und standardisiere sie (Pipeline). Teile die Daten zufällig in 70 % Training und 30 % Test (random_state=42). Speichere das Modell als modell und für die Testkunden die vorhergesagte Wahrscheinlichkeit in test['risiko'].
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

MERKMALE = ["tage_seit_kauf", "frequenz", "umsatz"]
train, test = train_test_split(train_df, test_size=0.3, random_state=42)
modell = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression())
modell.fit(train[MERKMALE], train["churn"])
test = test.assign(risiko=modell.predict_proba(test[MERKMALE])[:, 1])
pd.Series(modell[-1].coef_[0], index=MERKMALE).round(2)

**Plausibel?** Die Koeffizienten zeigen die Richtung: Mehr Tage seit dem letzten Kauf sollten das Risiko erhöhen (positiv), mehr Bestellungen sollten es senken (negativ). Ist das so?

### 4 Ist das Modell gut? Lift

**Ziel:** Prüfen, ob das Modell die Abwanderer besser findet als der Zufall.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Sortiere die Testkunden in test nach risiko absteigend und nimm die oberen 20 %. Berechne: die Abwanderungsrate in den oberen 20 %, die Abwanderungsrate aller Testkunden, den Lift (Rate oben geteilt durch Rate gesamt) und den Anteil aller Abwanderer, der in den oberen 20 % liegt.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
oben = test.nlargest(int(0.2 * len(test)), "risiko")
rate_oben, rate_alle = oben["churn"].mean(), test["churn"].mean()
print(f"Abwanderungsrate obere 20 %: {rate_oben:.0%}   alle: {rate_alle:.0%}   Lift: {rate_oben / rate_alle:.2f}")
print(f"Die oberen 20 % enthalten {oben['churn'].sum() / test['churn'].sum():.0%} aller Abwanderer "
      f"(zufällig: 20 %, bestenfalls: {min(1, 0.2 / rate_alle):.0%}).")

**Plausibel?** Ein Lift über 1 heisst: besser als Zufall. Achtung: Wenn fast alle abwandern, kann auch ein perfektes Modell keinen grossen Lift haben. Darum steht «bestenfalls» daneben.

### 5 Regel gegen Modell

**Ziel:** Eine einfache Handregel mit dem Modell vergleichen: «Mindestens drei Bestellungen, aber seit 60 Tagen still.»

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Wende auf die Testkunden in test die Handregel an: frequenz mindestens 3 und tage_seit_kauf mindestens 60. Wie viele Kunden erfüllen die Regel, und wie viele davon sind abgewandert (Rate)? Vergleiche mit gleich vielen Testkunden mit dem höchsten risiko.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
regel = test[(test["frequenz"] >= 3) & (test["tage_seit_kauf"] >= 60)]
modell_top = test.nlargest(len(regel), "risiko")
pd.DataFrame({"Kunden": [len(regel), len(modell_top)],
              "Abwanderungsrate": [regel["churn"].mean(), modell_top["churn"].mean()]},
             index=["Handregel", "Modell (gleich viele)"]).round(2)

**Plausibel?** Ist die Regel fast so gut wie das Modell? Das ist kein Misserfolg, sondern ein wertvolles Ergebnis: Eine Regel versteht jeder. Und: Die Regel sucht wertvolle Kunden, die still werden. Ist das dieselbe Frage?

### 6 Alle Kunden bewerten (Stichtag 31. August 2025)

**Ziel:** Das trainierte Modell auf heute anwenden: Wer wird in den nächsten 90 Tagen still? Das ist die Grundlage der Kampagne.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Berechne mit merkmale('2025-08-31') die Merkmale aller Kunden und mit modell.predict_proba die Spalte churn_risiko. Ergebnis: DataFrame churn_scores mit customer_id, tage_seit_kauf, frequenz, umsatz, churn_risiko.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
churn_scores = merkmale("2025-08-31")
churn_scores["churn_risiko"] = modell.predict_proba(churn_scores[MERKMALE])[:, 1]
churn_scores.sort_values("churn_risiko", ascending=False).head(10)

**Plausibel?** Haben Kunden mit vielen Tagen seit dem letzten Kauf ein hohes Risiko? Diskussion: Das Modell hat aus einem Sommerfenster gelernt, die Kampagne läuft Richtung Weihnachten. Was könnte das ändern?

In [ ]:
plausibel("churn_scores", churn_scores)

### 7 Speichern

In [ ]:
speichern(churn_scores, "churn_scores")

## Erweiterung (optional)

### E1 Entscheidungsbaum

**Ziel:** Ein kleiner Entscheidungsbaum ist leichter zu erklären. Ist er auch gleich gut?

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Trainiere auf train einen DecisionTreeClassifier mit max_depth=3 (random_state=0), zeichne den Baum mit plot_tree und berechne den Lift der oberen 20 % auf test wie in Schritt 4.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
from sklearn.tree import DecisionTreeClassifier, plot_tree
baum = DecisionTreeClassifier(max_depth=3, random_state=0).fit(train[MERKMALE], train["churn"])
plt.figure(figsize=(14, 5))
plot_tree(baum, feature_names=MERKMALE, filled=True, impurity=False, proportion=True)
plt.show()
r = test.assign(risiko_baum=baum.predict_proba(test[MERKMALE])[:, 1]).nlargest(int(0.2 * len(test)), "risiko_baum")
print(f"Lift Baum: {r['churn'].mean() / test['churn'].mean():.2f}")

**Plausibel?** Welche Merkmale nutzt der Baum oben? Würdet ihr der Geschäftsleitung lieber den Baum oder die Regression zeigen?

### E2 Lift-Kurve

**Ziel:** Den Lift nicht nur bei 20 %, sondern für jeden Anteil angeschriebener Kunden sehen.

**Prompt für Gemini** (kopieren und im Gemini-Fenster einfügen):
```text
Zeichne für test eine Kurve: x = Anteil der Kunden nach risiko sortiert (0 bis 100 %), y = Anteil der erfassten Abwanderer. Zeichne die Diagonale (Zufall) dazu.
```

In [ ]:
# Euer Code: fügt hier ein, was Gemini vorschlägt, und führt die Zelle aus (▶).

In [ ]:
#@title 💡 Lösung: nur wenn ihr nicht weiterkommt (Doppelklick zeigt den Code, ▶ führt ihn aus)
s = test.sort_values("risiko", ascending=False)
x = np.arange(1, len(s) + 1) / len(s)
y = s["churn"].cumsum() / s["churn"].sum()
plt.figure(figsize=(5, 4))
plt.plot(x, y, label="Modell")
plt.plot([0, 1], [0, 1], "--", label="Zufall")
plt.xlabel("Anteil angeschriebene Kunden"); plt.ylabel("Anteil erfasste Abwanderer"); plt.legend()
plt.show()

**Plausibel?** Je weiter die Kurve über der Diagonale liegt, desto mehr bringt das Modell.